# NB1 — Baselines Count et TF-IDF

Notebook des pipelines **P01 à P05**.

## Portée du notebook

Ce notebook regroupe les **baselines les plus propres et les plus défendables** pour commencer :
- un modèle probabiliste sur comptages ;
- des modèles linéaires discriminants sur TF-IDF ;
- une variante rapide avec `SGDClassifier`.

**Hypothèse de travail :**
- le split **train / test est déjà réalisé** ;
- les fichiers `train.csv` et `test.csv` se trouvent dans `../../data/` ;
- le fichier `test.csv` est **étiqueté** et contient donc bien la colonne cible `target`.

In [1]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow
# !pip install mlflow

from collections import OrderedDict

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import Normalizer

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    evaluate_sklearn_pipeline,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
    GensimMeanEmbeddingVectorizer,
    SentenceTransformerVectorizer,
)

seed_everything(42)
import mlflow


from mlflow_utils import (
    setup_mlflow_tracking,
    fit_evaluate_and_log_sklearn_pipeline,
)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB1_count_tfidf_baselines"
RESULTS_DIR = "../../outputs/NB1"

# Configuration MLflow
from pathlib import Path

MLFLOW_EXPERIMENT_NAME = "DT_NB1_count_tfidf_baselines"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB1_count_tfidf_baselines


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [3]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())

Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


In [4]:
pipelines = OrderedDict({
    "P01_Count_MultinomialNB": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 1), min_df=2)),
        ("clf", MultinomialNB(alpha=0.5)),
    ]),
    "P02_TFIDF_Unigram_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.95)),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight=None)),
    ]),
    "P03_TFIDF_UniBi_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0, class_weight=None)),
    ]),
    "P04_TFIDF_UniBi_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P05_TFIDF_UniBi_SGDLog": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", SGDClassifier(loss="log_loss", penalty="l2", alpha=1e-5, max_iter=3000, random_state=42)),
    ]),
})

In [5]:
resultats = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    metrics = fit_evaluate_and_log_sklearn_pipeline(
        name=nom_pipeline,
        estimator=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        notebook_name="NB1",
        family_name="count_tfidf_baselines",
        output_dir=RESULTS_DIR,
        log_model=MLFLOW_LOG_MODEL,
    )
    resultats.append(metrics)

results_df = round_results(pd.DataFrame(resultats))
display(results_df)


Entraînement -> P01_Count_MultinomialNB


Pipeline(steps=[('vect', CountVectorizer(min_df=2)),
                ('clf', MultinomialNB(alpha=0.5))])

--------------------------------------------------------------------------------
Entraînement -> P02_TFIDF_Unigram_LogReg


Pipeline(steps=[('vect', TfidfVectorizer(max_df=0.95, min_df=2)),
                ('clf', LogisticRegression(max_iter=2000))])

--------------------------------------------------------------------------------
Entraînement -> P03_TFIDF_UniBi_LogReg


...

Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement -> P05_TFIDF_UniBi_SGDLog


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('clf',
                 SGDClassifier(alpha=1e-05, loss='log_loss', max_iter=3000,
                               random_state=42))])

--------------------------------------------------------------------------------


pipeline,P01_Count_MultinomialNB,P02_TFIDF_Unigram_LogReg,P03_TFIDF_UniBi_LogReg,P04_TFIDF_UniBi_LinearSVC,P05_TFIDF_UniBi_SGDLog
train_accuracy,0.9291,0.9037,0.8986,0.9956,0.9963
train_precision_macro,0.8767,0.9319,0.9376,0.9957,0.9959
train_recall_macro,0.8958,0.7476,0.7301,0.9898,0.9918
train_f1_macro,0.8858,0.8011,0.7851,0.9927,0.9938
train_precision_weighted,0.9312,0.9100,0.9080,0.9956,0.9963
train_recall_weighted,0.9291,0.9037,0.8986,0.9956,0.9963
train_f1_weighted,0.9300,0.8909,0.8832,0.9956,0.9963
train_precision_class_0,0.9635,0.8970,0.8904,0.9956,0.9965
train_recall_class_0,0.9488,0.9961,0.9984,0.9991,0.9989
train_f1_class_0,0.9561,0.9439,0.9413,0.9973,0.9977


In [6]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans {RESULTS_DIR}")

Fichiers CSV/XLSX enregistrés dans ../../outputs/NB1
